# 01 — Data Cleaning

**Question.** Can we build a single tidy weekly panel of ACLED events across all six regions that is comparable over 2014–2025?

**Method.** Load the six regional weekly `.xlsx` files, concatenate, parse dates, filter to the common window, and persist a cleaned Parquet.

**Takeaway.** A single `df_weekly` covering 2014-01-01 to 2025-12-31 is saved to `data/derived/acled_clean.parquet`; a separate full-range Africa frame is kept for the long-run supplement.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path('/Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5')
DATA = PROJECT / 'data'
DERIVED = DATA / 'derived'
DERIVED.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT)

Project: /Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5


## Load the six regional weekly files

File names follow `<Region>_aggregated_data_up_to_week_of-*.xlsx`. We map each file to a canonical `REGION` label.

In [2]:
REGION_FILES = {
    'Africa':                 'Africa_aggregated_data_up_to_week_of-2026-03-21.xlsx',
    'Asia-Pacific':           'Asia-Pacific_aggregated_data_up_to_week_of-2026-03-28.xlsx',
    'Europe-Central-Asia':    'Europe-Central-Asia_aggregated_data_up_to_week_of-2026-03-28.xlsx',
    'Latin-America-Caribbean':'Latin-America-the-Caribbean_aggregated_data_up_to_week_of-2026-03-21.xlsx',
    'Middle-East':            'Middle-East_aggregated_data_up_to_week_of-2026-03-21.xlsx',
    'US-and-Canada':          'US-and-Canada_aggregated_data_up_to_week_of-2026-03-28.xlsx',
}

frames = {}
for region, fname in REGION_FILES.items():
    path = DATA / fname
    df = pd.read_excel(path)
    df['REGION'] = region
    frames[region] = df
    print(f'{region:25s} rows={len(df):>7d}  cols={df.shape[1]}')

df_all = pd.concat(frames.values(), ignore_index=True)
print('Combined:', df_all.shape)
df_all.head()

Africa                    rows= 268511  cols=13


Asia-Pacific              rows= 208289  cols=13


Europe-Central-Asia       rows= 118846  cols=13


Latin-America-Caribbean   rows= 171517  cols=13


Middle-East               rows= 145353  cols=13


US-and-Canada             rows=  22478  cols=13
Combined: (934994, 13)


,WEEK,REGION,COUNTRY,ADMIN1,EVENT_TYPE,SUB_EVENT_TYPE,EVENTS,FATALITIES,POPULATION_EXPOSURE,DISORDER_TYPE,ID,CENTROID_LATITUDE,CENTROID_LONGITUDE
0,2004-10-23,Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388
1,2005-04-23,Africa,Algeria,Adrar,Battles,Armed clash,1,0,NaN,Political violence,47.0,26.4839,-1.388
2,2005-06-25,Africa,Algeria,Adrar,Battles,Armed clash,1,14,NaN,Political violence,47.0,26.4839,-1.388
3,2008-12-13,Africa,Algeria,Adrar,Battles,Armed clash,1,3,NaN,Political violence,47.0,26.4839,-1.388
4,2009-04-18,Africa,Algeria,Adrar,Battles,Armed clash,1,2,NaN,Political violence,47.0,26.4839,-1.388


In [3]:
# Inspect columns and dtypes
print(df_all.dtypes)
print('\nColumns:', list(df_all.columns))

WEEK                   datetime64[us]
REGION                            str
COUNTRY                           str
ADMIN1                            str
EVENT_TYPE                        str
SUB_EVENT_TYPE                    str
EVENTS                          int64
FATALITIES                      int64
POPULATION_EXPOSURE           float64
DISORDER_TYPE                     str
ID                            float64
CENTROID_LATITUDE             float64
CENTROID_LONGITUDE            float64
dtype: object

Columns: ['WEEK', 'REGION', 'COUNTRY', 'ADMIN1', 'EVENT_TYPE', 'SUB_EVENT_TYPE', 'EVENTS', 'FATALITIES', 'POPULATION_EXPOSURE', 'DISORDER_TYPE', 'ID', 'CENTROID_LATITUDE', 'CENTROID_LONGITUDE']


## Parse dates, add `YEAR` / `MONTH`

In [4]:
df_all['WEEK'] = pd.to_datetime(df_all['WEEK'], errors='coerce')
df_all['YEAR'] = df_all['WEEK'].dt.year
df_all['MONTH'] = df_all['WEEK'].dt.month

# Coverage check per region
coverage = (df_all.groupby('REGION')['WEEK']
            .agg(['min','max','count'])
            .rename(columns={'count':'n_rows'}))
coverage

,min,max,n_rows
REGION,,,
Africa,1996-12-28,2026-03-21,268511
Asia-Pacific,2009-12-26,2026-03-28,208289
Europe-Central-Asia,2017-12-30,2026-03-28,118846
Latin-America-Caribbean,2017-12-30,2026-03-21,171517
Middle-East,2014-12-27,2026-03-21,145353
US-and-Canada,2019-12-28,2026-03-28,22478


## Keep a full-range Africa frame for the long-run supplement

Africa has coverage back to the late 1990s. We persist the Africa frame *before* filtering so Step 4b can use it.

In [5]:
df_africa_full = df_all.loc[df_all['REGION'] == 'Africa'].copy()
print('Africa full range:', df_africa_full['WEEK'].min(), '→', df_africa_full['WEEK'].max(),
      'rows=', len(df_africa_full))
df_africa_full.to_parquet(DERIVED / 'acled_africa_full.parquet', index=False)

Africa full range: 1996-12-28 00:00:00 → 2026-03-21 00:00:00 rows= 268511


## Filter to common window 2014-01-01 → 2025-12-31

US-and-Canada begins 2019 — that caveat will be surfaced in the report, but we do NOT drop US-and-Canada rows; we simply keep all regions in the same date window and flag the coverage gap.

In [6]:
mask = (df_all['WEEK'] >= '2014-01-01') & (df_all['WEEK'] <= '2025-12-31')
df_weekly = df_all.loc[mask].copy()
print('Filtered rows:', len(df_weekly), 'of', len(df_all))

df_weekly.groupby('REGION')['YEAR'].agg(['min','max','count'])

Filtered rows: 838798 of 934994


,min,max,count
REGION,,,
Africa,2014,2025,207859
Asia-Pacific,2014,2025,187590
Europe-Central-Asia,2017,2025,114321
Latin-America-Caribbean,2017,2025,166322
Middle-East,2014,2025,141171
US-and-Canada,2019,2025,21535


## Quick sanity checks

In [7]:
# Null counts on key columns
key_cols = ['WEEK','REGION','COUNTRY','EVENT_TYPE','SUB_EVENT_TYPE','DISORDER_TYPE','EVENTS','FATALITIES']
print(df_weekly[key_cols].isna().sum())

# Annual global totals
annual = (df_weekly.groupby('YEAR')[['EVENTS','FATALITIES']].sum().astype(int))
annual

WEEK              0
REGION            0
COUNTRY           0
EVENT_TYPE        0
SUB_EVENT_TYPE    0
DISORDER_TYPE     0
EVENTS            0
FATALITIES        0
dtype: int64


,EVENTS,FATALITIES
YEAR,,
2014,24662,47069
2015,36222,62546
2016,76458,124168
2017,114165,184837
2018,213135,178908
2019,231625,153678
2020,268297,139323
2021,293601,160264
2022,326622,172326


In [8]:
# Event-type and disorder-type value counts
print('EVENT_TYPE:')
print(df_weekly['EVENT_TYPE'].value_counts(dropna=False))
print('\nDISORDER_TYPE:')
print(df_weekly['DISORDER_TYPE'].value_counts(dropna=False))

EVENT_TYPE:
EVENT_TYPE
Protests                      322456
Violence against civilians    135421
Strategic developments        111854
Battles                       104890
Riots                          84112
Explosions/Remote violence     80065
Name: count, dtype: int64

DISORDER_TYPE:
DISORDER_TYPE
Political violence                    364690
Demonstrations                        358675
Strategic developments                111854
Political violence; Demonstrations      3579
Name: count, dtype: int64


## Persist cleaned frame

In [9]:
out_path = DERIVED / 'acled_clean.parquet'
df_weekly.to_parquet(out_path, index=False)
print('Wrote', out_path, '  size(MB)=', round(out_path.stat().st_size/1e6, 2))

Wrote /Users/zhangtingmin/Desktop/PhD/Year_1_Sem_2/POLI_3148/Assignment 1 copy 5/data/derived/acled_clean.parquet   size(MB)= 5.15


## Sanity cross-check against top-level country-year files

*Skipped.* The two reference files (`number_of_political_violence_events_by_country-year...` and `number_of_reported_fatalities...`) are not present in `data/`. Annual totals above are the authoritative series for this analysis.